# BranchLab — Train, verify, and explain a small language model

This notebook loads the published portfolio model, demonstrates generation and
same-weight cache measurements, and provides an optional exact experiment rerun.
Use a GPU runtime for training. Running the full study consumes your compute
quota; loading the release does not retrain. See the report for the precise data
subset, hardware, seeds, token budgets and limitations.


In [ ]:
# @title Set up the released project
import pathlib, subprocess, os
ROOT = pathlib.Path('/content/branchlab-project')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', 'v0.2.0',
                    'https://github.com/liyuqin606-del/branchlab.git', str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[dev]'], check=True)
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# @title Download the inference bundle and verify its checksum
import hashlib, urllib.request, tarfile
ASSET = 'branchlab-v0.2.0-inference.tar.gz'
BASE = 'https://github.com/liyuqin606-del/branchlab/releases/download/v0.2.0/'
urllib.request.urlretrieve(BASE + ASSET, ASSET)
urllib.request.urlretrieve(BASE + 'SHA256SUMS', 'SHA256SUMS')
expected = {line.split()[1]: line.split()[0] for line in pathlib.Path('SHA256SUMS').read_text().splitlines()}
assert hashlib.sha256(pathlib.Path(ASSET).read_bytes()).hexdigest() == expected[ASSET]
with tarfile.open(ASSET) as archive:
    archive.extractall('artifacts', filter='data')
print('Verified weights and tokenizer extracted.')


In [ ]:
# @title Generate with the small trained story model
from branchlab.cli import load_inference
from branchlab.training import resolve_device
DEVICE = resolve_device('auto')
MODEL_PATH = ROOT / 'artifacts/portfolio/showcase-gqa'
model, tokenizer = load_inference(MODEL_PATH, DEVICE)
prompt = 'Once upon a time, there was a little girl'
ids = torch.tensor([tokenizer.encode(prompt)], device=DEVICE)
generator = torch.Generator(device=DEVICE).manual_seed(42)
output = model.generate(ids, 128, eos_id=tokenizer.eos_id, cache_mode='static',
                        temperature=.8, top_k=40, generator=generator)
print(tokenizer.decode(output[0].tolist()))
print(f'{model.num_parameters():,} parameters, {model.config.n_heads} query / {model.config.kv_heads} KV heads')


In [ ]:
# @title Verify and measure the three cache paths on identical tokens
from branchlab.systems import benchmark_paths
import json
if DEVICE.type == 'cuda':
    model.to(dtype=torch.bfloat16)
encoded = tokenizer.encode(prompt)
fixed = (encoded * (64 // len(encoded) + 1))[:64]
result = benchmark_paths(model, torch.tensor([fixed], device=DEVICE), decode_tokens=64, repeats=7)
print(json.dumps(result, indent=2))


In [ ]:
# @title Measure CUDA Graph replay (CUDA only; reports capture cost)
if DEVICE.type == "cuda":
    subprocess.run(["python", "scripts/cuda_graph_decode.py", "--model", str(MODEL_PATH),
                    "--output", "artifacts/notebook-graph"], check=True)
else:
    print("CUDA Graph measurements require an NVIDIA GPU.")


## Read and test the implementation

Read `docs/ARCHITECTURE.md` beside `src/branchlab/model.py`. The manual attention
path explicitly shows projection shapes and causality; the optimized path uses
PyTorch SDPA. GQA is an existing architecture. The following tests check outputs,
gradients, multi-token cache updates and interrupted-training recovery.


In [ ]:
# @title Run meaningful correctness checks
subprocess.run(['python', '-m', 'pytest', 'tests/test_attention_extensions.py',
                'tests/test_portfolio.py', '-q'], check=True)


## Optional: rerun the frozen training study

This section downloads one ~249 MB official training shard and a ~10 MB
validation shard. It deterministically prepares 50,000 training documents and
512 development / 512 test documents. The main comparison is three paired
training seeds × MHA/GQA at equal token budgets; the independently declared
showcase uses a larger budget. It is not an equal-FLOP or full-data benchmark.
Check GPU availability and the synthetic preflight before running all jobs.
Periodic full-state checkpoints support continuation within the same files;
back them up to persistent storage before the Colab runtime expires.


In [ ]:
# @title Opt in to the optional full training study
RUN_TRAINING = False # @param {type:"boolean"}
print("Training enabled" if RUN_TRAINING else "Inference-only: optional training cells will be skipped")


In [ ]:
# @title Prepare pinned data (optional)
if RUN_TRAINING:
    subprocess.run(['branchlab', 'prepare-official', '--output', 'artifacts/official-data'], check=True)


In [ ]:
# @title Measure resource use before training (optional)
if RUN_TRAINING:
    subprocess.run(['python', 'scripts/run_portfolio.py', '--preflight',
                    '--output', 'artifacts/portfolio-repeat'], check=True)


In [ ]:
# @title Run the fixed study and generate a verified report (optional)
if RUN_TRAINING:
    subprocess.run(['python', 'scripts/run_portfolio.py', '--prepared', 'artifacts/official-data',
                    '--output', 'artifacts/portfolio-repeat', '--max-seconds', '7200'], check=True)
    subprocess.run(['python', 'scripts/report_portfolio.py', '--results', 'artifacts/portfolio-repeat',
                    '--prepared', 'artifacts/official-data', '--output', 'reports/portfolio-repeat'], check=True)
